# ViSceT5 — Pretrain **gen_all** (decoder read-scene-text, đòn bẩy #1)
Chạy tuần tự. `gen_all` = huấn luyện decoder **sinh scene-text** (khớp đúng đường finetune: encoder chỉ nhận câu hỏi + ảnh + OCR-feature) + MLM/ITM/TWC làm phụ trợ (×0.5) — phần pretrain trực tiếp có ích cho bộ sinh câu trả lời seq2seq.

Sau khi pretrain xong & upload lên HF, dùng `notebooks/finetune_colab.ipynb` để finetune từ nó.

In [ ]:
!git clone https://github.com/Kussssssss/ViSceT5.git
%cd ViSceT5
!git pull

In [ ]:
!bash setup.sh
# Nếu Colab báo cần restart: Runtime > Restart, rồi chạy tiếp TỪ cell cấu hình.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'hf_xxx'                       # <== ĐIỀN token HF của bạn
HF_PRETRAIN_REPO = 'Kus669/ViSceT5-pretrain-genall'     # repo sẽ lưu MODEL PRETRAIN (gen_all)

In [ ]:
import argparse
from scripts import prepare_dataset
prepare_dataset.main(argparse.Namespace(config='configs/data/ViTextVQA.yaml', data_dir='./datasets'))

In [ ]:
from scripts import init_model
init_model.main()

### 1) SMOKE TEST (verify 1 batch — chạy TRƯỚC)
Tìm trong log: `✅ [GEN] gen_loss finite & > 0` và `[GEN] gen_loss requires grad`. Phải truyền `args_list=[...]` (không dùng sys.argv) để mode có tác dụng trong kernel.

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--smoke_test', 'True',
])

### 2) FULL PRETRAIN gen_all
Chỉnh `--num_train_epochs` tuỳ nhu cầu. Theo dõi `loss_gen` trong log eval (giảm dần).

In [ ]:
import importlib
from training import pretrain
importlib.reload(pretrain)
pretrain.main(args_list=[
    'configs/pretrain.yaml',
    '--loss_ablation_mode', 'gen_all',
    '--num_train_epochs', '3',
])

### 3) Upload model pretrain lên HF (để finetune_colab.ipynb dùng)

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
api.create_repo(repo_id=HF_PRETRAIN_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path='/content/ViSceT5/output/pretrain', repo_id=HF_PRETRAIN_REPO,
                  repo_type='model', ignore_patterns=['optimizer.pt'])
print('Uploaded pretrain ->', HF_PRETRAIN_REPO)